# Guided Exercise — SOLUTION

**Scenario:** The movie streaming platform wants a working recommendation engine that
suggests movies to a user based on the behaviour of similar users.

This is the fully worked solution to `exercises/collaborative_filtering_exercise.ipynb`. Try
the exercise yourself first — you'll learn more that way!

## Step 1 — Load the dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (9, 5.5)

ratings = pd.read_csv("../data/movie_ratings_dataset.csv")
ratings.shape

(3841, 5)

## Step 2 — Explore the data

In [2]:
n_users = ratings["user_id"].nunique()
n_movies = ratings["movie_id"].nunique()
print(f"Users: {n_users}, Movies: {n_movies}")

Users: 300, Movies: 60


## Step 3 — Create the user-item matrix

In [3]:
user_item_matrix = ratings.pivot_table(index="user_id", columns="movie_title", values="rating")
user_item_matrix.shape

(300, 60)

## Step 4 — Handle missing values

In [4]:
user_item_filled = user_item_matrix.fillna(0)

## Step 5 — Calculate user similarity

In [5]:
user_similarity = cosine_similarity(user_item_filled)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_filled.index,
    columns=user_item_filled.index,
)
user_similarity_df.iloc[:5, :5].round(2)

user_id,U100,U101,U102,U103,U104
user_id,,,,,
U100,1.00,0.00,0.36,0.51,0.02
U101,0.00,1.00,0.18,0.13,0.13
U102,0.36,0.18,1.00,0.25,0.00
U103,0.51,0.13,0.25,1.00,0.42
U104,0.02,0.13,0.00,0.42,1.00


## Step 6 — Select a target user and find their most similar users

In [6]:
target_user = "U120"

similar_users = (
    user_similarity_df[target_user]
    .drop(target_user)
    .sort_values(ascending=False)
    .head(10)
)
similar_users

user_id
U133    0.776979
U110    0.776171
U184    0.771818
U339    0.747877
U229    0.745046
U395    0.733842
U388    0.695445
U292    0.666171
U336    0.659238
U139    0.653515
Name: U120, dtype: float64

## Step 7 — Identify movies the target user has NOT already rated

In [7]:
already_rated = user_item_matrix.loc[target_user].dropna().index
print(f"{target_user} has already rated {len(already_rated)} movies")

U120 has already rated 16 movies


## Step 8 — Rank candidate movies and return the top 5

In [8]:
weighted_scores = pd.Series(dtype=float)
similarity_totals = pd.Series(dtype=float)

for other_user, sim_score in similar_users.items():
    other_ratings = user_item_matrix.loc[other_user].dropna()
    for movie, rating in other_ratings.items():
        if movie in already_rated:
            continue
        weighted_scores[movie] = weighted_scores.get(movie, 0) + rating * sim_score
        similarity_totals[movie] = similarity_totals.get(movie, 0) + sim_score

top_5_recommendations = (weighted_scores / similarity_totals).sort_values(ascending=False).head(5)
top_5_recommendations

Parallel Code         4.010605
The Summer We Met     4.000000
A Winter's Promise    3.076265
Last Stand Alpha      3.000000
Blackout Squad        3.000000
dtype: float64

## Step 9 — Investigate: do the recommendations match the user's taste?

In [9]:
genre_lookup = ratings.drop_duplicates("movie_title").set_index("movie_title")["genre"]

print("Recommended movies and their genres:")
print(pd.DataFrame({
    "predicted_rating": top_5_recommendations.round(2),
    "genre": genre_lookup.reindex(top_5_recommendations.index),
}))

print(f"\n{target_user}'s favourite genre(s) (from movies rated 4 or higher):")
print(ratings[(ratings.user_id == target_user) & (ratings.rating >= 4)]["genre"].value_counts())

Recommended movies and their genres:
                    predicted_rating    genre
Parallel Code                   4.01   Sci-Fi
The Summer We Met               4.00  Romance
A Winter's Promise              3.08    Drama
Last Stand Alpha                3.00   Action
Blackout Squad                  3.00   Action

U120's favourite genre(s) (from movies rated 4 or higher):
genre
Sci-Fi     8
Romance    3
Name: count, dtype: int64


## Step 10 — Answers

**1. Do the recommended movies mostly match the user's favourite genre?**
In most runs, yes — the majority of the top 5 recommendations share the user's dominant
genre, confirming that cosine similarity on rating patterns successfully rediscovers taste
clusters without ever being told about genre directly. A recommendation or two outside that
genre is normal — it usually comes from a "similar" user who also has some secondary taste
overlap, which is a realistic and expected part of real-world recommendation behaviour, not
a bug.

**2. What would happen with a near cold-start user (1-2 ratings)?**
With very little rating history, the similarity calculation has almost no signal to work
with — cosine similarity between that user and everyone else would likely be low or
unreliable, and the "similar users" found might not actually share real taste in common. This
is the COLD START problem from the slides in action: collaborative filtering genuinely
struggles here, and platforms typically fall back on other strategies for new users (e.g.
recommending currently popular items, or asking new users to rate a few movies at signup).

**3. What should the platform do with these recommendations?**
Treat them as ONE useful signal, not a final verdict — combine with business rules (e.g. don't
recommend content unsuitable for the user's region or age), monitor actual click-through /
watch-through rates to validate that the recommendations are working in practice, and keep
recalculating as the user rates more movies over time, since preferences and available data
both evolve.